# Day 10 — max & argmax  ·  *the itch for softmax*

**Milestone 2 (attention):** before we build the smooth "soft" selection that attention needs, we look at the **hard** version — `max`/`argmax` — and feel exactly why it's not good enough. That gap motivates everything in M2.

## 1. Review (~5 min)

Recall **day 3** (dot product) and **day 7** (cosine similarity).

In [ ]:
import numpy as np
rng = np.random.default_rng(10)

**R1.** (day 3) the dot product $a\cdot b$.

In [ ]:
def r_dot(a, b):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    for _ in range(4):
        n = int(rng.integers(2, 8)); a = rng.normal(size=n); b = rng.normal(size=n)
        assert np.allclose(fn(a, b), np.dot(a, b))
    print("✅ Review R1 passed")

check_r1(r_dot)

**R2.** (day 7) cosine similarity.

In [ ]:
def r_cosine_similarity(a, b):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    for _ in range(4):
        n = int(rng.integers(2, 8)); a = rng.normal(size=n); b = rng.normal(size=n)
        assert np.allclose(fn(a, b), np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b)))
    print("✅ Review R2 passed")

check_r2(r_cosine_similarity)

## 2. Lecture note (~10 min): hard selection and why it hurts

**The itch.** Attention has to *choose* which things to focus on; a classifier has to *pick* a label. The obvious tool is **max** (the biggest value) and **argmax** (its index). But hard picking has two problems that block learning.

**The picture.** `argmax` is a spotlight on the single largest entry — equivalently a **one-hot** vector: a 1 at the winner, 0 everywhere else.

**The tools.** $\max(s)=\text{largest value}$, $\arg\max(s)=\text{its index}$.

**Knobs & effect — the two problems.**
1. **Discontinuous.** Nudge the runner-up just past the leader and the output *jumps* — no smooth in-between. 
2. **Not differentiable.** Because of that jump, there's no useful gradient, so a model can't learn *how* to shift its attention. 

That's the whole motivation for softmax: a **soft, differentiable** stand-in for argmax (days 12–14).

**In the wild.** Greedy decoding (argmax over the vocabulary), "hard" attention, and the final prediction of a classifier — all argmax, all non-differentiable, which is exactly why training uses the soft version instead.

## 3. Exercises (~15 min)

### Exercise 1 — argmax
Return the **index** of the largest entry of a 1-D array `s`.

In [ ]:
def my_argmax(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    for _ in range(6):
        s = rng.normal(size=int(rng.integers(2, 10)))
        assert fn(s) == int(np.argmax(s))
    print("✅ Exercise 1 passed")

check_e1(my_argmax)

### Exercise 2 — the one-hot picture
Return the **one-hot** vector for the argmax: same length as `s`, a `1.0` at the largest entry and `0.0` elsewhere.

In [ ]:
def one_hot_argmax(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    for _ in range(5):
        s = rng.normal(size=int(rng.integers(2, 8)))
        v = np.asarray(fn(s))
        assert v.shape == s.shape and v.sum() == 1.0
        assert v[int(np.argmax(s))] == 1.0
    print("✅ Exercise 2 passed")

check_e2(one_hot_argmax)

### Exercise 3 — feel the discontinuity (the effect)
Walk from vector `a` to vector `b` along $s(t)=(1-t)\,a + t\,b$ for each `t` in `ts`, and return the list of `argmax(s(t))`. You'll see it stays constant then **jumps** — never smooth. That jump is why argmax has no gradient.

In [ ]:
def argmax_path(a, b, ts):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    a = np.array([1.0, 0.0, 0.0]); b = np.array([0.0, 0.0, 1.0])
    ts = np.linspace(0, 1, 11)
    out = list(fn(a, b, ts))
    oracle = [int(np.argmax((1 - t) * a + t * b)) for t in ts]
    assert out == oracle
    assert len(set(out)) > 1, "the winner should jump somewhere along the path"
    print("✅ Exercise 3 passed")

check_e3(argmax_path)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_dot(a, b):
    return float(np.sum(a * b))

def ref_cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def ref_my_argmax(s):
    return int(np.argmax(s))

def ref_one_hot_argmax(s):
    v = np.zeros_like(np.asarray(s, dtype=float))
    v[int(np.argmax(s))] = 1.0
    return v

def ref_argmax_path(a, b, ts):
    return [int(np.argmax((1 - t) * a + t * b)) for t in ts]

check_r1(ref_dot)
check_r2(ref_cosine_similarity)
check_e1(ref_my_argmax)
check_e2(ref_one_hot_argmax)
check_e3(ref_argmax_path)
print("\nAll reference solutions pass. 🎉  See you on day 11!")